In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import dill

In [3]:
df = pd.read_csv('./data/preprocessed_reviews_shuffled.csv')
print(df.head())
# Display all unique labels in the first column
# Remove rows where the first column has the label 'sentiment'
df = df[df.iloc[:, 0] != 'sentiment']

# Display the updated counts for each unique label in the first column
print(df.iloc[:, 0].value_counts())

   positive      very satisfied with the quality 
0   neutral  delivery was fine product is decent 
1  negative                  not worth the price 
2   neutral  delivery was fine product is decent 
3  negative                  not worth the price 
4  negative     late delivery and poor packaging 
positive
positive    9977
negative    9937
neutral     5085
Name: count, dtype: int64


In [4]:


# Using the preprocessed reviews DataFrame (df) already loaded

# Map sentiment labels to numerical values
# Assuming df.iloc[:, 0] contains the sentiment labels ('positive', 'negative', 'neutral')
sentiment_mapping = {'positive': 1, 'negative': 0, 'neutral': 2}
df['sentiment_numerical'] = df.iloc[:, 0].map(sentiment_mapping)

# 2. Split the dataset into training and testing sets
# Assuming df.iloc[:, 1] contains the review text
X_train, X_test, y_train, y_test = train_test_split(
    df.iloc[:, 1], # Review text
    df['sentiment_numerical'], # Numerical sentiment labels
    test_size=0.25,
    random_state=42,
    stratify=df['sentiment_numerical'] # Stratify to maintain class distribution
)

# 3. Convert text data into numerical features using TF-IDF
# lowercase=True automatically converts text to lowercase to normalize it
vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 4. Train a Logistic Regression classifier
model = LogisticRegression(max_iter=1000) # Increased max_iter for convergence
model.fit(X_train_tfidf, y_train)

# 5. Evaluate the model performance
predictions = model.predict(X_test_tfidf)
print("--- Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, predictions, target_names=sentiment_mapping.keys()))



--- Model Evaluation ---
Accuracy: 100.00%

Classification Report:
               precision    recall  f1-score   support

    positive       1.00      1.00      1.00      2484
    negative       1.00      1.00      1.00      2495
     neutral       1.00      1.00      1.00      1271

    accuracy                           1.00      6250
   macro avg       1.00      1.00      1.00      6250
weighted avg       1.00      1.00      1.00      6250



In [6]:
# 6. Test the model with brand new custom text
new_reviews = [
    "I am very happy with how fast this arrived!",
    "Worst experience ever, I want a refund.",
    "It's okay, not bad but not great either.",
    "Horrible product",
    "awesome product"
]

# Transform the new text using the already fitted vectorizer
new_reviews_tfidf = vectorizer.transform(new_reviews)
new_predictions = model.predict(new_reviews_tfidf)

# Reverse mapping for display
reverse_sentiment_mapping = {v: k for k, v in sentiment_mapping.items()}

print("\n--- Custom Predictions ---")
for review, prediction in zip(new_reviews, new_predictions):
    sentiment_label = reverse_sentiment_mapping[prediction]
    print(f"Review: \"{review}\" -> Predicted Sentiment: {sentiment_label.capitalize()}")


--- Custom Predictions ---
Review: "I am very happy with how fast this arrived!" -> Predicted Sentiment: Positive
Review: "Worst experience ever, I want a refund." -> Predicted Sentiment: Neutral
Review: "It's okay, not bad but not great either." -> Predicted Sentiment: Positive
Review: "Horrible product" -> Predicted Sentiment: Neutral
Review: "awesome product" -> Predicted Sentiment: Neutral


```markdown
## Using BERT-based Sentence Transformers for Feature Extraction

To leverage the power of pre-trained language models like BERT, we can use a sentence transformer to convert our review texts into dense vector embeddings. These embeddings capture semantic meaning and can often lead to better performance compared to traditional methods like TF-IDF, especially for tasks like sentiment analysis.

First, we need to install the `sentence-transformers` library.
```

In [12]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained sentence transformer model
# 'all-MiniLM-L6-v2' is a good general-purpose model
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Re-split the dataset (using the original df before numerical mapping if applicable)
# Ensure df.iloc[:, 0] are labels and df.iloc[:, 1] are texts, and they are clean
X = df.iloc[:, 1] # Review text
y = df['sentiment_numerical'] # Numerical sentiment labels

X_train_text, X_test_text, y_train_bert, y_test_bert = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Encode text data into embeddings using the BERT model
print("Generating BERT embeddings for training data...")
X_train_embeddings = bert_model.encode(X_train_text.tolist(), show_progress_bar=True)
print("Generating BERT embeddings for test data...")
X_test_embeddings = bert_model.encode(X_test_text.tolist(), show_progress_bar=True)

print(f"Shape of X_train_embeddings: {X_train_embeddings.shape}")
print(f"Shape of X_test_embeddings: {X_test_embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Generating BERT embeddings for training data...


Batches:   0%|          | 0/586 [00:00<?, ?it/s]

Generating BERT embeddings for test data...


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Shape of X_train_embeddings: (18749, 384)
Shape of X_test_embeddings: (6250, 384)


```markdown
Now that we have the BERT embeddings, we can train a new Logistic Regression model using these features. This will replace the TF-IDF based model.
```

In [14]:
# 4. Train a Logistic Regression classifier with BERT embeddings
print("Training Logistic Regression model with BERT embeddings...")
bert_lr_model = LogisticRegression(max_iter=1000, solver='liblinear') # 'liblinear' often works well for smaller datasets
bert_lr_model.fit(X_train_embeddings, y_train_bert)

# 5. Evaluate the BERT-based model performance
bert_predictions = bert_lr_model.predict(X_test_embeddings)
print("\n--- BERT Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test_bert, bert_predictions) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test_bert, bert_predictions, target_names=sentiment_mapping.keys()))

# 6. Test the BERT-based model with brand new custom text
new_reviews_bert = [
    "I am very happy with how fast this arrived!",
    "Worst experience ever, I want a refund.",
    "It's okay, not bad but not great either.",
    "Horrible product",
    "awesome product",
    "okay product need improvements"
]

# Transform the new text using the already fitted BERT model
print("\nGenerating BERT embeddings for custom reviews...")
new_reviews_embeddings = bert_model.encode(new_reviews_bert, show_progress_bar=True)
new_bert_predictions = bert_lr_model.predict(new_reviews_embeddings)

print("\n--- Custom Predictions (BERT Model) ---")
for review, prediction in zip(new_reviews_bert, new_bert_predictions):
    sentiment_label = reverse_sentiment_mapping[prediction]
    print(f"Review: \"{review}\" -> Predicted Sentiment: {sentiment_label.capitalize()}")

Training Logistic Regression model with BERT embeddings...


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(



--- BERT Model Evaluation ---
Accuracy: 100.00%

Classification Report:
               precision    recall  f1-score   support

    positive       1.00      1.00      1.00      2484
    negative       1.00      1.00      1.00      2495
     neutral       1.00      1.00      1.00      1271

    accuracy                           1.00      6250
   macro avg       1.00      1.00      1.00      6250
weighted avg       1.00      1.00      1.00      6250


Generating BERT embeddings for custom reviews...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- Custom Predictions (BERT Model) ---
Review: "I am very happy with how fast this arrived!" -> Predicted Sentiment: Positive
Review: "Worst experience ever, I want a refund." -> Predicted Sentiment: Negative
Review: "It's okay, not bad but not great either." -> Predicted Sentiment: Neutral
Review: "Horrible product" -> Predicted Sentiment: Negative
Review: "awesome product" -> Predicted Sentiment: Positive
Review: "okay product need improvements" -> Predicted Sentiment: Neutral


In [15]:
sys.stderr.write(f"\n saving bert model pls wait ....\n")
model_filename = f"./models/BertClassifier_pkl_model.pkl"
with open(model_filename, "wb") as model_file:
    dill.dump(bert_lr_model, model_file)
sys.stderr.write(f"\nModel saved successfully to: {model_filename}\n")


 saving bert model pls wait ....

Model saved successfully to: ./models/BertClassifier_pkl_model.pkl


68